In [15]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import torch.optim as optim
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib.patches import Patch


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error

from models.lstm import embedded_LSTM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# data

In [2]:
# Define the custom color palette (Rust-Red, Orange, Gold, Tan)
nfl_colors = ['#a83216', '#d8631d', '#e29726', '#b29d83']

# Apply the custom color cycle and palette globally
sns.set_palette(sns.color_palette(nfl_colors))
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=nfl_colors)

from utils.dataprocessing import train_test_split


df = pd.read_parquet("hf://datasets/afzalmengal/nfl-4th-downs-dataset/data/train-00000-of-00001.parquet")

df['play_type'] = df['play_type'].replace({'pass': 'go', 'run': 'go', 'qb_kneel': 'no_play'})

valid_plays = ['go', 'field_goal', 'punt']
df = df[df['play_type'].isin(valid_plays)].copy()


In [3]:
df.columns

Index(['game_id', 'season', 'home_team', 'away_team', 'posteam', 'defteam',
       'qtr', 'game_seconds_remaining', 'yardline_100', 'ydstogo',
       'goal_to_go', 'score_differential', 'home_score', 'total_home_score',
       'away_score', 'total_away_score', 'wp', 'def_wp', 'spread_line',
       'total_line', 'play_type', 'posteam_type', 'home_coach', 'away_coach',
       'epa', 'wpa', 'success', 'fourth_down_converted', 'fourth_down_failed',
       'label', 'week', 'season_type', 'off_Division_weekly',
       'off_Wins_weekly', 'off_Losses_weekly', 'off_Ties_weekly',
       'off_League_Rank_weekly', 'off_Division_Rank_weekly',
       'def_Division_weekly', 'def_Wins_weekly', 'def_Losses_weekly',
       'def_Ties_weekly', 'def_League_Rank_weekly', 'def_Division_Rank_weekly',
       'off_Wins_yearly', 'off_Losses_yearly', 'off_Ties_yearly',
       'off_Division_Placement_yearly', 'off_Division_Name_yearly',
       'off_Points_For_yearly', 'off_Points_For_Per_Game_yearly',
       'off_

In [13]:


# Enforce formatting
torch.set_printoptions(precision=2, sci_mode=False)

# 1. Define Features
continuous_features = [
    'score_differential', 'ydstogo', 'yardline_100', 'game_seconds_remaining', 
    'spread_line', 'total_line', 'wp'
]
categorical_features = ['posteam', 'defteam', 'posteam_type', 'goal_to_go', 'qtr', 'season_type']
target_action = 'play_type'
target_wpa = 'wpa'


def extract_tensors(df):
    X_cont = torch.tensor(df[continuous_features].values, dtype=torch.float32)
    X_cat = torch.tensor(df[categorical_features].values, dtype=torch.long)
    A = torch.tensor(df[target_action].values, dtype=torch.long)
    Y = torch.tensor(df[target_wpa].values, dtype=torch.float32)
    return X_cont, X_cat, A, Y


# Drop any rows containing NaNs in our required columns to prevent scaler/tensor crashes
all_required_cols = continuous_features + categorical_features + [target_action, target_wpa]
df_clean = df.dropna(subset=all_required_cols).copy()

# Filter out 'no_play' 
valid_plays = ['go', 'field_goal', 'punt']
df_clean = df_clean[df_clean[target_action].isin(valid_plays)].copy()

# Val-Train / Test Split (80% Train, 20% Test)
# Train / Val Split (80% Train, 20% Val)
train_df0, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df0, test_size=0.2, random_state=42)

# Scale Continuous Features. FIT only on the training data to prevent future leakage, then TRANSFORM both
scaler = StandardScaler()
train_df[continuous_features] = scaler.fit_transform(train_df[continuous_features])
val_df[continuous_features] = scaler.transform(val_df[continuous_features])
test_df[continuous_features] = scaler.transform(test_df[continuous_features])

# 5. Encode Categorical Features
# Convert strings (like team names) into integers required by nn.Embedding
encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    # Fit on the full clean dataset to ensure all unique teams/quarters are mapped
    le.fit(df_clean[col].astype(str)) 
    train_df[col] = le.transform(train_df[col].astype(str))
    val_df[col] = le.transform(val_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    encoders[col] = le

# Encode Target Action (go, field_goal, punt -> 0, 1, 2)
target_encoder = LabelEncoder()
target_encoder.fit(df_clean[target_action].astype(str))
train_df[target_action] = target_encoder.transform(train_df[target_action].astype(str))
val_df[target_action] = target_encoder.transform(val_df[target_action].astype(str))
test_df[target_action] = target_encoder.transform(test_df[target_action].astype(str))

# 6. Extract Tensors for train, val and test
X_cont_train, X_cat_train, A_train, Y_train = extract_tensors(train_df)
X_cont_val, X_cat_val, A_val, Y_val = extract_tensors(val_df)
X_cont_test, X_cat_test, A_test, Y_test = extract_tensors(test_df)


print(f"Training shape: {X_cont_train.shape[0]} plays")
print(f"Testing shape: {X_cont_test.shape[0]} plays")

Training shape: 68547 plays
Testing shape: 21422 plays


In [18]:
cat_emb_dims = []
for col in categorical_features:
    # Find exactly how many unique items exist in this column
    num_classes = len(encoders[col].classes_)
    
    # Rule of thumb: emb_size is half the number of classes (min 2, max 16)
    emb_size = max(2, min(16, num_classes // 2)) 
    
    cat_emb_dims.append((num_classes, emb_size))

print("Dynamically calculated embedding dimensions:", cat_emb_dims)


# 1. Instantiate the class
model = embedded_LSTM(
    cat_emb_dims=cat_emb_dims, 
    num_continuous=7, 
    hidden_size=64
).to(device)

# Execute Training
model.fit(
    X_cont=X_cont_train, 
    X_cat=X_cat_train, 
    A=A_train, 
    Y=Y_train,
    X_cont_val=X_cont_val,  
    X_cat_val=X_cat_val,    
    A_val=A_val,            
    Y_val=Y_val,           
    epochs=10, 
    batch_size=64, 
    lr=0.001
)


Dynamically calculated embedding dimensions: [(32, 16), (32, 16), (2, 2), (2, 2), (6, 3), (2, 2)]
Starting training on cpu for 10 epochs...
Epoch [1/10] - Train Loss: 0.4014 | Val Loss: 0.3001
Epoch [2/10] - Train Loss: 0.2821 | Val Loss: 0.2744
Epoch [3/10] - Train Loss: 0.2610 | Val Loss: 0.2603
Epoch [4/10] - Train Loss: 0.2491 | Val Loss: 0.2548
Epoch [5/10] - Train Loss: 0.2420 | Val Loss: 0.2512
Epoch [6/10] - Train Loss: 0.2366 | Val Loss: 0.2462
Epoch [7/10] - Train Loss: 0.2333 | Val Loss: 0.2466
Epoch [8/10] - Train Loss: 0.2293 | Val Loss: 0.2447
Epoch [9/10] - Train Loss: 0.2260 | Val Loss: 0.2475
Epoch [10/10] - Train Loss: 0.2227 | Val Loss: 0.2443


In [19]:

X_cont_test = X_cont_test.to(device)
X_cat_test = X_cat_test.to(device)

# Generate predictions using the custom predict method
pred_actions, pred_wpa = model.predict(X_cont_test, X_cat_test)

# Move everything back to the CPU to use with Scikit-Learn
pred_actions_cpu = pred_actions.cpu().numpy()
pred_wpa_cpu = pred_wpa.cpu().numpy()
true_actions_cpu = A_test.cpu().numpy()
true_wpa_cpu = Y_test.cpu().numpy()

# =====================================================================
# CLASSIFICATION METRICS (Play Type Decision)
# =====================================================================
print("\n" + "="*50)
print("CLASSIFICATION METRICS (ACTION DECISION)")
print("="*50)

# Retrieve the original string names from your target_encoder
target_names = target_encoder.classes_

accuracy = accuracy_score(true_actions_cpu, pred_actions_cpu)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

print("Detailed Classification Report:")
print(classification_report(true_actions_cpu, pred_actions_cpu, target_names=target_names))

# =====================================================================
# REGRESSION METRICS (Win Probability Added)
# =====================================================================
print("\n" + "="*50)
print("REGRESSION METRICS (WIN PROBABILITY ADDED)")
print("="*50)

# Calculate standard regression errors
mae = mean_absolute_error(true_wpa_cpu, pred_wpa_cpu)
mse = mean_squared_error(true_wpa_cpu, pred_wpa_cpu)
rmse = np.sqrt(mse)

print(f"Mean Absolute Error (MAE):      {mae:.4f}")
print(f"Mean Squared Error (MSE):       {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")


CLASSIFICATION METRICS (ACTION DECISION)
Overall Accuracy: 90.23%

Detailed Classification Report:
              precision    recall  f1-score   support

  field_goal       0.88      0.88      0.88      5077
          go       0.75      0.63      0.68      3145
        punt       0.94      0.97      0.96     13200

    accuracy                           0.90     21422
   macro avg       0.86      0.83      0.84     21422
weighted avg       0.90      0.90      0.90     21422


REGRESSION METRICS (WIN PROBABILITY ADDED)
Mean Absolute Error (MAE):      0.0336
Mean Squared Error (MSE):       0.0033
Root Mean Squared Error (RMSE): 0.0570


In [ ]:
model_save_path = 'fourth_down_lstm_weights.pth'

#torch.save(model.state_dict(), model_save_path)